# 09 · Round 2: finalists, final models and test

In round 1 (08) the model taken to the final stage was the search's best candidate by its search score. Trained again with three seeds, that candidate scored 0.586 ± 0.016 (scene-balanced fresh-burn IoU in FP16) on the search split it had been picked on, against 0.685 in the search. A search score is one shortened run with one seed on a few scene groups, and the best of 150 of them is picked partly for a lucky run: it overstates its candidate, as the best of many noisy measurements always does (the winner's curse).

Round 2 replaces that choice with a finalist stage (PyNAS 1.1, `docs/DESIGN.md` §5.1), fixed before any of it ran:

- **finalists**: the 8 candidates with the best search fitness; finalist 1 is the round-1 winner;
- each **trained as a final model**: seeds 0, 1 and 2, 40,000 batches, FP16 with distillation and BatchNorm folded for the last 20%;
- **compared on the validation sets** (`heo_val`, `phisat2_val`), which take no part in the search: the mean over seeds of the scene-balanced fresh-burn IoU in FP16, with its standard error;
- finalists **within one standard error of the best** count as equal; of those, the one with the **lowest predicted latency** is taken.

The chosen model then goes through the same final stage as in 08 (three seeds, the same baselines, both ablations), is exported, and the test split is read once for this round. The round-1 test results were read before this round was planned; the reason for it is the search-split measurement above, not the test. They stay as they are and are reported next to these. Runs already done with the same settings in round 1 (the round-1 winner's three seeds, the baselines) are copied with a `reused.json` saying from where, and not trained again.

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
for p in (os.path.join(os.getcwd(), "pyqnas", "src"), os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
import pynas
from pynas import config, final, report, runner
from pynas.hardware import myriad

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
CONFIG = Path("configs/burnscar.ini")
cfg = config.load(CONFIG)
cfg2 = config.with_overrides(cfg, {"Final.finalists": "8"})
BASE = Path(cfg.get("Run", "out"))
SEARCH, ROUND1, R2 = BASE / "search", BASE / "final", BASE / "final_r2"
TEACHER_Q = BASE / "teacher" / "runs" / "queue.json"
GPUS = list(range(torch.cuda.device_count())) or [0]
print(f"PyNAS {pynas.__version__}; {cfg2.getint('Final', 'finalists')} finalists, seeds {cfg2.get('Final', 'seeds')}, "
      f"{cfg2.getint('Final', 'steps')} batches")

## Finalists

The 8 candidates and their search scores, then their final training in the background (running the cell again carries on).

In [ ]:
# [2] the finalists, trained like final models
display(final.finalists(cfg2, SEARCH)[["gen", "score", "latency_ms", "ir_mb", "params", "spec"]])
FQ = final.finalist_queue(cfg2, SEARCH, R2, reuse=[ROUND1 / "runs"])
Q = runner.read_queue(FQ)
print(f"{len(Q['runs'])} runs, {sum((Path(Q['out']) / r['run'] / 'reused.json').exists() for r in Q['runs'])} copied from round 1")
runner.launch(FQ, gpus=GPUS)

In [ ]:
# [3] progress every 15 minutes (interrupting stops the watching, not the training)
runner.wait(FQ, every=900)

Each finalist: its search score, the search split measured again after the final training (mean over seeds: the difference is the winner's curse), and the validation score with its standard error, against predicted latency. The pick follows the rule above.

In [ ]:
# [4] the pick
pick, table = final.pick_finalist(FQ, cfg2, SEARCH, R2)
print(json.dumps({k: v for k, v in pick.items() if k != "table"}, indent=1, default=str))
display(table[["label", "search_score", "search_remeasured", "val_mean", "val_se", "within_one_se", "latency_ms", "ir_mb"]].round(4))
fig, ax = plt.subplots(figsize=(8, 5))
report.plot_finalists(table, ax)
plt.tight_layout(); plt.savefig(R2 / "finalists.png", dpi=200); plt.show()

## Final stage of the pick

Three seeds (from the finalist stage), the baselines (from round 1) and both ablations of the pick, scored on the search and validation sets.

In [ ]:
# [5] the final runs (those already done are copied)
Q2F = final.queue(cfg2, SEARCH, R2, reuse=[ROUND1 / "runs"])
Q2 = runner.read_queue(Q2F)
print(f"{len(Q2['runs'])} runs, {sum((R2 / 'runs' / r['run'] / 'reused.json').exists() for r in Q2['runs'])} copied")
runner.launch(Q2F, gpus=GPUS)

In [ ]:
# [6] progress every 15 minutes
runner.wait(Q2F, every=900)

In [ ]:
# [7] fresh-burn IoU in FP16 on the search and validation sets: mean and standard deviation over seeds
report.final_table(Q2F, cfg2).round(3)

In [ ]:
# [8] the deployed model of round 2
dep = final.deploy(Q2F, cfg2, "models/onboard", "burnscar_pynas_r2")
print(json.dumps({k: v for k, v in dep.items() if k not in ("layers", "ir_vs_pytorch")}, indent=1, default=str))
if dep.get("ir_vs_pytorch"):
    display(pd.DataFrame(dep["ir_vs_pytorch"]).set_index("set"))

In [ ]:
# [9] the hardware numbers of the pick and the baselines (one seed each)
from pynas.hardware.latency import latency_ms
from pynas.space.network import n_params_used
res = final.results(Q2F)
one = res[res.role.isin(["final", "baseline"])].drop_duplicates("label")
nets = {f"m{i}": runner.load_result(R2 / "runs" / r.run) for i, r in enumerate(one.itertuples())}
shape = tuple(json.load(open(R2 / "runs" / one.run.iloc[0] / "run.json"))["input_shape"])
hw = myriad.check(nets, R2 / "hardware", shape, cfg2) if myriad.status(cfg2)["model_optimizer"] else {}
lat = latency_ms(nets, shape, cfg2) if myriad.status(cfg2)["nnmeter"] else {}
pd.DataFrame([dict(model=r.label, params_M=n_params_used(nets[k], shape) / 1e6, ir_mb=hw.get(k, {}).get("ir_mb"),
                   compiled=hw.get(k, {}).get("compiled"), latency_ms=lat.get(k))
              for k, r in zip(nets, one.itertuples())]).set_index("model").round(3)

## Test

Read once for round 2 (kept in `models/pynas/final_r2/test.json`), then shown with round 1, whose stored result is read back.

In [ ]:
# [10] the test split, read once for round 2, with round 1
scores, per_gsd = final.test(Q2F, cfg2, teacher_queue=TEACHER_Q)
s1, g1 = final.test(ROUND1 / "runs" / "queue.json", cfg, teacher_queue=TEACHER_Q)
dep1 = json.load(open("models/onboard/burnscar_pynas/deploy.json"))
both = pd.concat([s1[~s1.label.str.startswith("teacher")].assign(label="round 1: " + s1.label),
                  s1[s1.run == dep1["run"]].assign(label="round 1: deployed"),
                  scores[~scores.label.str.startswith("teacher")].assign(label="round 2: " + scores.label),
                  scores[scores.run == dep["run"]].assign(label="round 2: deployed"),
                  scores[scores.label.str.startswith("teacher")]])
display(report.test_table(both, cfg2).round(3))

In [ ]:
# [11] fresh-burn IoU per test set and resolution (round 2)
ORDER = ["winner", "smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8", "smp:Unet:mobilenet_v2:depth=4:decoder=64-32-16-8",
         "smp:Unet:efficientnet-b0", "winner, no_kd", "winner, fp32", "teacher"]
report.plot_test_iou(per_gsd, labels=ORDER)
plt.tight_layout(); plt.savefig(R2 / "test_iou_by_gsd.png", dpi=200); plt.show()

In [ ]:
# [12] the test scenes: the deployed models of both rounds, the slimmed EfficientNet-B0 U-Net and the teacher
from pynas import teacher
from pynas.train import data as D
spec = D.DataSpec.from_config(cfg2)
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
eb0 = res[(res.label == "smp:Unet:efficientnet-b0:depth=4:decoder=64-32-16-8") & (res.seed == 0)].run.iloc[0]
models = {f"PyNAS round 2 (seed {dep['seed']})": (runner.load_result(R2 / "runs" / dep["run"], dev), "fp16"),
          f"PyNAS round 1 (seed {dep1['seed']})": (runner.load_result(ROUND1 / "runs" / dep1["run"], dev), "fp16"),
          "U-Net EfficientNet-B0, slim": (runner.load_result(R2 / "runs" / eb0, dev), "fp16"),
          "teacher (mean of seeds)": (final.Mean(teacher.models(TEACHER_Q, dev)), "fp32")}
for name in spec.test:
    s = D.open_set(spec, name, teacher=False)
    fig = report.plot_scenes(s, report.predict_set(models, spec, s, dev), spec, name, max_scenes=3)
    fig.savefig(R2 / f"test_scenes_{name}.png", dpi=200); plt.show()

Next: the ArgOS test suite on the deployed model of round 2, `models/onboard/burnscar_pynas_r2/`, and the model on the Myriad X itself.